In [3]:
# Cell 1 - Environment check and imports
import pandas as pd
import numpy as np
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

df = pd.read_csv('/content/bq-results-20260323-203006-1774297820102.csv')
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.shape[1]}")
print(f"Readmission rate: {df['readmitted_30d'].mean():.4f}")

TensorFlow: 2.19.0
GPU available: []

Dataset shape: (177353, 53)
Columns: 53
Readmission rate: 0.1760


In [4]:
# Cell 2 - Data diagnosis
print(f"Total rows: {len(df)}")
print(f"Unique patients: {df['subject_id'].nunique()}")
print(f"Date range: {df['admittime'].min()} to {df['admittime'].max()}")
print(f"Readmission rate: {df['readmitted_30d'].mean():.4f}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Total rows: 177353
Unique patients: 78358
Date range: 2109-12-14 22:50:00 to 2214-12-15 19:11:00
Readmission rate: 0.1760

Missing values:
marital_status                4422
language                       242
insurance                     2030
admission_location               1
discharge_location           23078
num_lab_tests_24h            27314
num_abnormal_labs            27314
hemoglobin_min               40004
wbc_max                      40552
creatinine_max               43327
sodium_min                   45300
sodium_max                   45300
potassium_min                44274
potassium_max                44274
glucose_min                  46739
glucose_max                  46739
days_since_last_discharge    70035
days_to_next_admission       78358
readmitted_90d                   1
dtype: int64


In [5]:
# Cell 3 - Preprocessing: handle missing values and prepare features
# Fill missing values
df['marital_status'] = df['marital_status'].fillna('UNKNOWN')
df['language'] = df['language'].fillna('UNKNOWN')
df['insurance'] = df['insurance'].fillna('UNKNOWN')
df['admission_location'] = df['admission_location'].fillna('UNKNOWN')
df['discharge_location'] = df['discharge_location'].fillna('UNKNOWN')

# Lab values - fill with median (clinical standard)
lab_cols = ['num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min',
            'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max',
            'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max']
for col in lab_cols:
    df[col] = df[col].fillna(df[col].median())

# Historical features - fill with 0 (first-time patients)
hist_cols = ['days_since_last_discharge', 'num_admissions_last_30d',
             'num_admissions_last_90d', 'num_admissions_last_year',
             'total_prior_admissions', 'recent_admission_flag',
             'frequent_flyer_flag']
for col in hist_cols:
    df[col] = df[col].fillna(0)

# Fill remaining single missing values with median
df = df.fillna(df.median(numeric_only=True))

print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")

Missing values remaining: 0
Shape: (177353, 53)


In [6]:
# Cell 4 - Encode categorical features and define feature columns
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['gender', 'marital_status', 'language', 'insurance',
                    'admission_location', 'discharge_location', 'admission_type']

for col in categorical_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

# Define feature columns (drop IDs, timestamps, targets)
drop_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime',
             'readmitted_30d', 'readmitted_60d', 'readmitted_90d',
             'days_to_next_admission']

feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"Feature columns ({len(feature_cols)}):")
print(feature_cols)

Feature columns (45):
['gender', 'age', 'race', 'marital_status', 'language', 'insurance', 'admission_type', 'admission_location', 'discharge_location', 'los_hours', 'los_days', 'num_diagnoses', 'cci_mi', 'cci_chf', 'cci_pvd', 'cci_cvd', 'cci_dementia', 'cci_copd', 'cci_diabetes', 'cci_ckd', 'cci_cancer', 'num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min', 'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max', 'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max', 'num_medications', 'polypharmacy_flag', 'anticoagulant_flag', 'insulin_flag', 'opioid_flag', 'antibiotic_flag', 'num_admissions_last_30d', 'num_admissions_last_90d', 'num_admissions_last_year', 'days_since_last_discharge', 'total_prior_admissions', 'recent_admission_flag', 'frequent_flyer_flag']


In [7]:
# Cell 5 - Drop race, finalize feature columns
feature_cols = [c for c in feature_cols if c != 'race']
print(f"Final feature columns ({len(feature_cols)}):")
print(feature_cols)

Final feature columns (44):
['gender', 'age', 'marital_status', 'language', 'insurance', 'admission_type', 'admission_location', 'discharge_location', 'los_hours', 'los_days', 'num_diagnoses', 'cci_mi', 'cci_chf', 'cci_pvd', 'cci_cvd', 'cci_dementia', 'cci_copd', 'cci_diabetes', 'cci_ckd', 'cci_cancer', 'num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min', 'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max', 'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max', 'num_medications', 'polypharmacy_flag', 'anticoagulant_flag', 'insulin_flag', 'opioid_flag', 'antibiotic_flag', 'num_admissions_last_30d', 'num_admissions_last_90d', 'num_admissions_last_year', 'days_since_last_discharge', 'total_prior_admissions', 'recent_admission_flag', 'frequent_flyer_flag']


In [8]:
# Cell 6 - Build patient sequences for LSTM (fixed - per admission labels)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import StandardScaler

MAX_SEQ_LEN = 10

# Sort by patient and admission time
df['admittime'] = pd.to_datetime(df['admittime'])
df = df.sort_values(['subject_id', 'admittime']).reset_index(drop=True)

# Scale features
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# Build sequences per patient
# For each admission, use all PREVIOUS admissions as context
# Label = that specific admission's readmitted_30d
X_sequences = []
y_labels = []
subject_ids = []

for subject_id, group in df.groupby('subject_id'):
    group = group.reset_index(drop=True)
    features = group[feature_cols].values
    labels = group['readmitted_30d'].values

    # For each admission in this patient's history
    for i in range(len(group)):
        # Sequence = admissions up to and including current
        seq = features[:i+1]
        label = labels[i]
        X_sequences.append(seq)
        y_labels.append(label)
        subject_ids.append(subject_id)

# Pad sequences
X_padded = pad_sequences(X_sequences, maxlen=MAX_SEQ_LEN,
                         dtype='float32', padding='pre', truncating='pre')
y_array = np.array(y_labels)

print(f"Total sequences: {len(X_padded)}")
print(f"Sequence shape: {X_padded.shape}")
print(f"Readmission rate: {y_array.mean():.4f}")

Total sequences: 177353
Sequence shape: (177353, 10, 44)
Readmission rate: 0.1760


In [9]:
print(df['admittime'].min())
print(df['admittime'].max())
print(df['admittime'].median())

2109-12-14 22:50:00
2214-12-15 19:11:00
2154-08-09 02:13:00


In [10]:
# Find percentile-based cutoff dates
sorted_dates = patient_first_admit['first_admittime'].sort_values()
train_cutoff = sorted_dates.quantile(0.70)
val_cutoff = sorted_dates.quantile(0.85)
print(f"Train cutoff (70%): {train_cutoff}")
print(f"Val cutoff (85%):   {val_cutoff}")

NameError: name 'patient_first_admit' is not defined

In [11]:
# Cell 7 - Chronological train/val/test split (admission level)
# Get admittime for each sequence (use original df order)
df_sorted = df.sort_values(['subject_id', 'admittime']).reset_index(drop=True)
admittimes = df_sorted['admittime'].values

train_cutoff = pd.Timestamp('2168-05-07')
val_cutoff = pd.Timestamp('2180-05-13')

train_idx = admittimes < train_cutoff
val_idx = (admittimes >= train_cutoff) & (admittimes < val_cutoff)
test_idx = admittimes >= val_cutoff

X_train, y_train = X_padded[train_idx], y_array[train_idx]
X_val, y_val = X_padded[val_idx], y_array[val_idx]
X_test, y_test = X_padded[test_idx], y_array[test_idx]

print(f"Train: {X_train.shape} | Readmission rate: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape} | Readmission rate: {y_val.mean():.4f}")
print(f"Test:  {X_test.shape} | Readmission rate: {y_test.mean():.4f}")

Train: (118374, 10, 44) | Readmission rate: 0.1759
Val:   (26371, 10, 44) | Readmission rate: 0.1702
Test:  (32608, 10, 44) | Readmission rate: 0.1814


In [12]:
# Cell 8 - Build Bidirectional LSTM with Bahdanau Attention
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Bidirectional, LSTM,
                                      Dense, Dropout, Layer)
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# Bahdanau Attention Layer
class BahdanauAttention(Layer):
    def __init__(self, units=64, **kwargs):
        super(BahdanauAttention, self).__init__(**kwargs)
        self.W = Dense(units)
        self.V = Dense(1)

    def call(self, encoder_output):
        # Score each time step
        score = self.V(tf.nn.tanh(self.W(encoder_output)))
        # Convert scores to weights (must sum to 1)
        attention_weights = tf.nn.softmax(score, axis=1)
        # Weighted sum of all time steps
        context_vector = attention_weights * encoder_output
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# Build model
def build_lstm_attention_model(seq_len, n_features):
    inputs = Input(shape=(seq_len, n_features))

    # Bidirectional LSTM
    lstm_out = Bidirectional(LSTM(64, return_sequences=True,
                                   dropout=0.3))(inputs)

    # Attention
    context_vector, attention_weights = BahdanauAttention(64)(lstm_out)

    # Dense layers
    x = Dense(32, activation='relu')(context_vector)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=output)
    return model

model = build_lstm_attention_model(MAX_SEQ_LEN, len(feature_cols))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 44)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 10, 128)        │        55,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bahdanau_attention              │ [(None, 128), (None,   │         8,321 │
│ (BahdanauAttention)             │ 10, 1)]                │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,290 (266.76 KB)

 Trainable params: 68,290 (266.76 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Cell 9 - Compile and train the model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np

# Handle class imbalance - give more weight to positive cases
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)
class_weight = {0: 1.0, 1: neg/pos}
print(f"Class weight for positive class: {neg/pos:.2f}")

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['AUC']
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_AUC',
    patience=5,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_AUC',
    factor=0.5,
    patience=3,
    mode='max',
    verbose=1
)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Class weight for positive class: 4.69
Epoch 1/30
463/463 ━━━━━━━━━━━━━━━━━━━━ 36s 65ms/step - AUC: 0.6712 - loss: 1.0649 - val_AUC: 0.6826 - val_loss: 0.6385 - learning_rate: 0.0010
Epoch 2/30
463/463 ━━━━━━━━━━━━━━━━━━━━ 30s 64ms/step - AUC: 0.6860 - loss: 1.0504 - val_AUC: 0.6871 - val_loss: 0.6136 - learning_rate: 0.0010
Epoch 3/30
463/463 ━━━━━━━━━━━━━━━━━━━━ 30s 64ms/step - AUC: 0.6903 - loss: 1.0459 - val_AUC: 0.6897 - val_loss: 0.6230 - learning_rate: 0.0010
Epoch 4/30
463/463 ━━━━━━━━━━━━━━━━━━━━ 30s 65ms/step - AUC: 0.6935 - loss: 1.0424 - val_AUC: 0.6923 - val_loss: 0.6105 - learning_rate: 0.0010
Epoch 5/30
463/463 ━━━━━━━━━━━━━━━━━━━━ 30s 64ms/step - AUC: 0.6974 - loss: 1.0389 - val_AUC: 0.6920 - val_loss: 0.6165 - learning_rate: 0.0010
Epoch 6/30
463/463 ━━━━━━━━━━━━━━━━━━━━ 29s 64ms/step - AUC: 0.7005 - loss: 1.0360 - val_AUC: 0.6940 - val_loss: 0.6047 - learning_rate: 0.0010
Epoch 7/30
463/463 ━━━━━━━━━━━━━━━━━━━━ 29s 63ms/step - AUC: 0.7005 - loss: 1.0354 - val_AUC: 0.69

In [14]:
# Cell 10 - Evaluate on test set
from sklearn.metrics import roc_auc_score

y_pred_proba = model.predict(X_test, verbose=0).flatten()
test_auroc = roc_auc_score(y_test, y_pred_proba)

print(f"Test AUROC: {test_auroc:.4f}")
print(f"\n--- Model Comparison ---")
print(f"Logistic Regression:       0.7021")
print(f"LR + Class Weights:        0.7025")
print(f"LR + SMOTE:                0.6984")
print(f"XGBoost (tuned):           0.7197")
print(f"LSTM + Attention:          {test_auroc:.4f}")
print(f"Target:                    0.7500")

Test AUROC: 0.6951

--- Model Comparison ---
Logistic Regression:       0.7021
LR + Class Weights:        0.7025
LR + SMOTE:                0.6984
XGBoost (tuned):           0.7197
LSTM + Attention:          0.6951
Target:                    0.7500


In [15]:
# Cell 11 - Save model
model.save('/content/lstm_attention_model.h5')
print("Model saved.")

Model saved.
